In [1]:
!pip install -q fastparquet
!pip install -q MLForecast
!pip install -q utilsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, RollingMin, RollingMax, ExponentiallyWeightedMean, ExpandingMean, ExpandingStd, SeasonalRollingMean, SeasonalRollingStd
import lightgbm as lgb

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [3]:
import warnings
warnings.filterwarnings('ignore', message='Found null values in')

In [4]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

# данные о праздниках 
holidays = pd.read_excel('/kaggle/input/datasets/faibus/diploma/holidays.xlsx')
holidays['Holidays'] = pd.to_datetime(holidays['Holidays'], format='%Y.%m.%d')

# данные о погоде
weather = pd.read_csv('/kaggle/input/datasets/faibus/diploma/weather.csv')
weather['final_temp'] = (weather['temperature_ekb'] * 0.14) + (weather['temperature_msk'] * 0.22) + (weather['temperature_nov'] * 0.11) + (weather['temperature_ros'] * 0.16) + (weather['temperature_sam'] * 0.15) + (weather['temperature_spb'] * 0.09) + (weather['temperature_tve'] * 0.13)
weather = weather[['date', 'final_temp']]
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')

# данные о днях промо
promo_days = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/promo_days.parquet', engine='fastparquet')

# данные о характеристиках SKU
sku_charasterictics = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/sku_features.parquet', engine='fastparquet')[['SKU_id', 'Category4', 'Brand', 'SupplierID']]

In [5]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'lumpy' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(912713, 3)

In [6]:
# джойним характеристики SKU
real_demand = pd.merge(real_demand, sku_charasterictics, left_on = ['unique_id'], right_on = ['SKU_id'])

# создаём дни праздников (корректнее сказать "признак выходных")
real_demand['is_holiday'] = real_demand['ds'].isin(holidays['Holidays'])

# джойним промо-дни
real_demand = pd.merge(real_demand, promo_days, how = 'left', left_on = ['ds', 'unique_id'], right_on = ['date', 'SKU_id'])

# джойним погодные условия (температуру)
real_demand = pd.merge(real_demand, weather, how = 'left', left_on = ['ds'], right_on = ['date'])

# заполяем пропуски в промо-днях значением False
real_demand['is_promo'] = real_demand['is_promo'].fillna(False)

# оставляем только нужные колонки
real_demand = real_demand[['unique_id', 'ds', 'y', 'is_holiday', 'is_promo','final_temp', 'Category4', 'Brand', 'SupplierID']].copy()

# меняем тип данных bool -> int
real_demand['is_holiday'] = real_demand['is_holiday'].astype(int)
real_demand['is_promo'] = real_demand['is_promo'].astype(int)

real_demand.shape

/tmp/ipykernel_55/3268695253.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  real_demand['is_promo'] = real_demand['is_promo'].fillna(False)


(912713, 9)

### Применяем Label Encoding для категориальных колонок

In [7]:
# Кодируем в категории
static_cols = ['Category4', 'Brand', 'SupplierID']
for col in static_cols:
    real_demand[col] = real_demand[col].astype('category')

real_demand.shape

(912713, 9)

### Подготовка ряда к обучению

In [8]:
### Фильтрация рядов, достаточно длинных для обучения

horizon = 14

real_demand_filtered = filter_long_series(
    real_demand,
    horizon = 14,
    n_val_windows = 1,
    n_test_windows = 3,
    min_train_points = 35,
    id_col="unique_id",
)



### Разделение на train, val, test

train_df, val_df, test_windows_df = split_train_val_test(
    real_demand_filtered,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    id_col="unique_id",
    ds_col="ds",
)

### Обучение

In [9]:
models = {
    'lgb_mae_base': lgb.LGBMRegressor(
        objective='mae',
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        verbosity=-1,
    ),
}

mlf = MLForecast(
    models=models, 
    freq='D',
    
    lags=[14, 15, 16, 17, 18, 19, 20, 21, 28],

    lag_transforms={
        14: [
            # Средние (разные окна → модель учит тренд из их разницы)
            RollingMean(window_size=7),
            RollingMean(window_size=14),
            RollingMean(window_size=28),
            
            # Волатильность (вместе со средними → модель учит CV)
            RollingStd(window_size=7),
            RollingStd(window_size=14),
            
            # Экстремумы (вместе → модель учит range/амплитуду)
            RollingMin(window_size=14),
            RollingMax(window_size=14),
            
            # EWM
            ExponentiallyWeightedMean(alpha=0.3),
            ExponentiallyWeightedMean(alpha=0.1),
            
            # Долгосрочный уровень и волатильность
            ExpandingMean(),
            ExpandingStd(),
        ],
        21: [
            RollingMean(window_size=7),
            RollingMean(window_size=14),
            RollingStd(window_size=7),
        ],
        # Сезонность по дню недели: среднее и std "каждого понедельника/вторника/..."
        # season_length=7 → группирует по дню недели
        # window_size=4 → за последние 4 таких же дня (4 недели)
        7: [
            SeasonalRollingMean(season_length=7, window_size=4),
            SeasonalRollingStd(season_length=7, window_size=4),
        ],
    },
    
    date_features=['month', 'day', 'dayofweek', 'week'],
)


# Обучение
static_features = ['Category4', 'Brand', 'SupplierID']
exog_cols = ['is_holiday', 'is_promo', 'final_temp']
mlf.fit(
    train_df,
    static_features=static_features,
    dropna=False,
)


# Предсказание на окнах
def predict_one_window_frozen_ml(mlf, history_df, target_window_df, horizon=14):
    # 1) оставляем id с полным окном 14 дней
    target_counts = target_window_df.groupby("unique_id")["ds"].nunique()
    full_target_ids = target_counts[target_counts == horizon].index
    hist_ids = pd.Index(history_df["unique_id"].unique())
    eligible_ids = hist_ids.intersection(full_target_ids)
    if len(eligible_ids) == 0:
        raise ValueError("Нет рядов с полным покрытием target-окна.")
    history_part = history_df[history_df["unique_id"].isin(eligible_ids)].copy()
    target_part = target_window_df[target_window_df["unique_id"].isin(eligible_ids)].copy()
    
    # 2) MLForecast ожидает future exog в X_df (без y)
    X_df = target_part[["unique_id", "ds"] + exog_cols].drop_duplicates(["unique_id", "ds"])
    
    # 3) Predict без re-fit: используем new_df=history_part
    preds = mlf.predict(
        h=horizon,
        new_df=history_part,
        X_df=X_df,
    ).reset_index(drop=True)
    
    # 4) Склейка с фактом
    out = target_part[["unique_id", "ds", "y"]].merge(
        preds, on=["unique_id", "ds"], how="inner"
    )
    return out

### Прогнозы

In [14]:
# Обертка
def predict_fn(history_df, target_window_df, horizon):
    return predict_one_window_frozen_ml(
        mlf=mlf,
        history_df=history_df,
        target_window_df=target_window_df,
        horizon=horizon,
    )

# Сбор предсказаний
val_pred, test_pred = evaluate_frozen_windows(
    predict_one_window_fn=predict_fn,
    train_df=train_df,
    val_df=val_df,
    test_windows_df=test_windows_df,
    horizon=horizon,
    ds_col="ds",
)

### Расчет метрик

In [15]:
# -------- VAL metrics --------
val_model_cols = get_model_columns(val_pred, reserved_columns = ("unique_id", "ds", "cutoff", "y", "index", "test_window"))
val_metrics_per_window = compute_metrics_per_window(
    cv_results=val_pred,
    model_columns=val_model_cols,
    metrics_dict=DEFAULT_METRICS
)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)


# -------- TEST metrics --------
test_model_cols = get_model_columns(test_pred, reserved_columns = ("unique_id", "ds", "cutoff", "y", "index", "test_window"))
test_metrics_per_window = compute_metrics_per_window(
    cv_results=test_pred,
    model_columns=test_model_cols,
    metrics_dict=DEFAULT_METRICS
)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)
print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
lgb_mae_base,0.88698,2.208691,159.562673,82.319663



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
lgb_mae_base,0.936318,2.637289,164.602737,84.717706


In [16]:
test_pred.to_csv("ml_forecast_lumpy_test_pred.csv")